In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from collections import defaultdict

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h3

# ─── Parameters ──────────────────────────────────────────────────────────────
API_BASE      = 'https://wiig.dia.fi.upm.es/b4c_api'
CITY_ID       = 2        # 1=Madrid (no trips), 2=Barcelona, etc.
H3_RESOLUTION = 7        # 7 ≈ 1.2 km cells → ~20-40 bubbles; 8 ≈ 0.5 km → denser
MIN_VOLUME    = 50       # Drop hexes with fewer total trips than this
BUBBLE_SCALE  = 1.0      # Radius multiplier — tune visually after first run
BASE_RADIUS   = 18       # Base scatter size before volume scaling
OUTPUT_DIR    = Path('frontend/public/landing')
FIG_W, FIG_H  = 6, 5
DPI           = 150
COLOR         = '#027A76'
BG_COLOR      = '#FBF6EF'
BBOX_COLOR    = '#003849'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR.resolve()}')

In [ ]:
# City info — center coords + available generation types
city = requests.get(f'{API_BASE}/cities/{CITY_ID}').json()
center_lat = city['center_lat']
center_lon = city['center_lon']
print(f'City: {city["name"]}  center: ({center_lat:.4f}, {center_lon:.4f})')

# Pick first available generation_type from traffic combinations
combos = city.get('available_modes', {}).get('traffic_combinations', [])
if not combos:
    raise ValueError(f'City {CITY_ID} has no trip data — try a different CITY_ID')
generation_type = combos[0]['generation_type']
print(f'Using generation_type: {generation_type}')

# Fetch O/D hex flows from API
od = requests.get(
    f'{API_BASE}/cities/{CITY_ID}/trips/od-flows',
    params={'generation_type': generation_type, 'resolution': H3_RESOLUTION},
).json()
features = od.get('features', [])
print(f'O/D flow features: {len(features)}')

# Aggregate flow counts into per-hex volumes (origins + destinations)
hex_volumes: dict = defaultdict(int)
for f in features:
    count = int(f['properties']['count'])
    hex_volumes[f['properties']['orig_hex']] += count
    hex_volumes[f['properties']['dest_hex']] += count

# Filter by MIN_VOLUME, build DataFrame with hex centroid coordinates
rows = []
for cell, vol in hex_volumes.items():
    if vol >= MIN_VOLUME:
        clat, clon = h3.cell_to_latlng(cell)
        rows.append({'lat': clat, 'lon': clon, 'volume': vol})

bubbles_df = pd.DataFrame(rows)
print(f'Hexes after MIN_VOLUME={MIN_VOLUME} filter: {len(bubbles_df)}')
if not bubbles_df.empty:
    print(bubbles_df['volume'].describe().to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
ax.set_facecolor(BG_COLOR)
fig.patch.set_facecolor(BG_COLOR)

# Faint study-area bbox rectangle (10×10 km around city center)
lat_off = 5000 / 111320
lon_off = 5000 / (111320 * np.cos(np.radians(center_lat)))
rect_lons = [
    center_lon - lon_off, center_lon + lon_off,
    center_lon + lon_off, center_lon - lon_off,
    center_lon - lon_off,
]
rect_lats = [
    center_lat - lat_off, center_lat - lat_off,
    center_lat + lat_off, center_lat + lat_off,
    center_lat - lat_off,
]
ax.plot(rect_lons, rect_lats,
        color=BBOX_COLOR, linewidth=0.4, alpha=0.25, linestyle='--', zorder=1)

# Gradient circles: 4 stacked scatter passes per point (core → halo)
if not bubbles_df.empty:
    max_vol = bubbles_df['volume'].max()
    radii = BASE_RADIUS * BUBBLE_SCALE * np.sqrt(bubbles_df['volume'].values / max_vol)
    lons  = bubbles_df['lon'].values
    lats  = bubbles_df['lat'].values

    for mult, alpha in [(1.0, 0.85), (2.0, 0.30), (3.5, 0.12), (5.5, 0.05)]:
        ax.scatter(lons, lats, s=(radii * mult) ** 2,
                   c=COLOR, alpha=alpha, linewidths=0, zorder=3)

# Clip axes to bbox + 15% margin
ax.set_xlim(center_lon - lon_off * 1.15, center_lon + lon_off * 1.15)
ax.set_ylim(center_lat - lat_off * 1.15, center_lat + lat_off * 1.15)
ax.axis('off')
plt.tight_layout(pad=0)

out_path = OUTPUT_DIR / 'map_traffic_od.png'
fig.savefig(out_path, dpi=DPI, bbox_inches='tight', facecolor=BG_COLOR)
plt.show()
plt.close(fig)
print(f'✅ Saved to {out_path.resolve()}')
